# Importation des modules

In [1]:
import pandas as pd
import numpy as np
import os
import statsmodels

In [4]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Chargement des données
diviser les dépenses de santé par le PIB pour les avoir en %  
rajouter les données de dépenses de santé pour 1994 - 2004 en regardant sur les données de l'ocde  
rajouter la base mortalite  
faire le tri dans les pays de la bdd nb de médecins  


In [8]:
def load_data() :
    #PIB, pop_65, pop_tot, densite_medicale, out_of_pocket, mortalite :  #à remplacer par load_data() quand ce sera codé

    depenses_sante = pd.read_excel(os.path.join("data", "Dépenses_santé_en_volume.xlsx"))
    pib = pd.read_excel(os.path.join("data", "PIB.xlsx"))
    pop_tot = pd.read_excel(os.path.join("data", "Population_tot.xlsx"))
    pop65 = pd.read_excel(os.path.join("data", "Population_+65ans.xlsx"))
    
    return depenses_sante, pib, pop_tot, pop65

depenses_sante, pib, pop_tot, pop65 = load_data()
depenses_sante

/opt/python/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/python/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/python/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,Dépenses de santé pour des fonctions choisies ...,NaN,NaN,NaN,NaN,NaN
5,Ouvrir la page produit,Ouvrir dans le Data Browser,NaN,NaN,NaN,NaN
6,Description:,-,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN
8,Dernière mise à jour des données:,NaN,NaN,22/01/2026 23:00,NaN,NaN
9,Dernière modification de la structure de données:,NaN,NaN,22/01/2026 23:00,NaN,NaN


# Construction du Time-to-Death

In [5]:
def time_to_death(mortalite, pop) :
    # pour pop, à voir si on prend pop_tot, pop_65 ou une population plus âgée encore
    # ou faire une somme pondérée des mortalités pour les 65-70, 70-75, etc
    # fait par chatgpt, à revoir en fonction de la structure des données
    df = mortalite.merge(
        pop_tot,
        on=["country", "year", "age"],
        how="inner"
    )

    df["weighted_mortality"] = df["mortality_rate"] * df["population"]
    ttd = (
        df.groupby(["country", "year"])["weighted_mortality"]
          .sum()
          .reset_index(name="TTD")
    )
    return ttd

#pop = pop_65   # ou pop_80, ou pop_tot...
#ttd = time_to_death(mortalite, pop)

# Mise en forme du panel

In [7]:
def panel(depenses_sante, pib, pop_65, pop_tot, densite_medicale, oop, ttd) :
    #construction du panel contenant toutes les données dont on a besoin dans un seul panel
    panel = (
        depenses_sante.merge(pop_65, on=["country", "year"])
           .merge(pib, on=["country", "year"])
           .merge(densite_medicale, on=["country", "year"])
           .merge(oop, on=["country", "year"])
           .merge(ttd, on=["country", "year"])
    )

    panel = panel.sort_values(["country", "year"])
    panel.set_index(["country", "year"], inplace=True)
    return panel

#panel = panel(depenses_sante, pib, pop_65, pop_tot, densite_medicale, oop, ttd)

# Régression et GMM